# Baseline experiments
This notebook implements the baseline Naive RAG architecture, representing the industry standard for processing unstructured legal data. The primary objective is to establish a performance benchmark using the "Omgevingswet" (legislation) and associated case law (jurisprudence). Furthermore, this notebook evaluates two distinct chunking strategies, fixed size and structure aware to determine the optimal configuration for legal document retrieval, as measured by the RAGAs framework.

## Imports

In [10]:
import os
import time
import pandas as pd
import json
from datasets import Dataset
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI

from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
from ragas.llms import llm_factory
import nest_asyncio
from ragas import aevaluate 
from ragas import RunConfig



C:\Users\verkad004\AppData\Local\Temp\ipykernel_43256\3276267353.py:16: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_43256\3276267353.py:16: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_43256\3276267353.py:16: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' inst

## Environment variables

In [11]:
env_path = os.path.join("..", ".env")
load_dotenv(dotenv_path=env_path)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
embedding_deployment = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

# Initialize Azure AD token provider
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

## Data Loading
To establish a foundation for the baseline experiments, both the legislation and jurisprudence datasets are integrated into a single, unified corpus. By concatenating these sources, the RAG system treats all legal information as a single knowledge base, providing a naive starting point.

In [12]:
legislation_path = '../data/legislation_omgevingswet_cleaned.jsonl'
jurisprudence_path = '../data/jurisprudence_omgevingswet_cleaned.jsonl'

common_cols = ['id', 'title', 'text', 'word_count', 'source_type']
df_legislation = pd.read_json(legislation_path, lines=True)
df_jurisprudence = pd.read_json(jurisprudence_path, lines=True)

df_leg_sub = df_legislation[common_cols]
df_jur_sub = df_jurisprudence[common_cols]

df_baseline = pd.concat([df_leg_sub, df_jur_sub], ignore_index=True)

print(f"Legislation loaded: {len(df_legislation)} rows")
print(f"Jurisprudence loaded: {len(df_jurisprudence)} rows")
print(f"Total documents for baseline: {len(df_baseline)}")

Legislation loaded: 725 rows
Jurisprudence loaded: 3404 rows
Total documents for baseline: 4129


In [13]:
# one list with all documents
documents = []

# loop through baseline DataFrame 
for _, row in df_baseline.iterrows():
    # Make new document
    doc = Document(
        page_content=str(row['text']),
        metadata={
            "id": row['id'],
            "title": row['title'],
            "source_type": row['source_type']
        }
    )
    documents.append(doc)

print(f"{len(documents)} documents made.")
print(f"Example metadata first document: {documents[0].metadata}")

4129 documents made.
Example metadata first document: {'id': 'Artikel 1.1', 'title': 'Omgevingswet - Artikel 1.1', 'source_type': 'legislation'}


## Chunking experiments
Niet alleen chunking size maar ook overlap

Kijken naar literatuur

### Fixed size chunking
Fixed-size chunking remains the most common approach due to its ease of implementation and predictable indexing performance (Nguyen et al., 2025; Bhat et al., 2025). However, a significant limitation is its lack of semantic awareness, which can result in "cutting" legal concepts or mixing unrelated ideas within a single chunk (Jimeno-Yepes et al., 2024; Kshirsagar, 2024). In the context of the Omgevingswet, this could mean that a crucial legal condition is separated from its parent article, potentially leading to incomplete or unsafe legal advice (Lu et al., 2025).

In [14]:
# Define splitter
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,      # Average size for balance between context and precision
    chunk_overlap=100,    # Remain context between chunks
    length_function=len,  
    separators=["\n\n", "\n", " ", ""] # Priority for cutting
)

# Apply splitter to documents
chunks_fixed = fixed_splitter.split_documents(documents)

# Validation of output
print(f"Strategy 1: Fixed-Size Results")
print(f"Total documents: {len(documents)}")
print(f"Total chunks created: {len(chunks_fixed)}")

# Sample
sample_idx = 0 
print(f"\n Sample Chunk {sample_idx}")
print(f"Metadata: {chunks_fixed[sample_idx].metadata}")
print(f"Content snippet: {chunks_fixed[sample_idx].page_content[:200]}...")

Strategy 1: Fixed-Size Results
Total documents: 4129
Total chunks created: 79725

 Sample Chunk 0
Metadata: {'id': 'Artikel 1.1', 'title': 'Omgevingswet - Artikel 1.1', 'source_type': 'legislation'}
Content snippet: Artikel 1.1 (begripsbepalingen) 1 De bijlage bij deze wet bevat begripsbepalingen voor de toepassing van deze wet en de daarop berustende bepalingen. 2 Begripsbepalingen die zijn opgenomen in een bijl...


### Structure-Aware Chunking
To preserve the logical integrity of legal texts, this study evaluates structure-aware chunking. This method uses natural document boundaries—such as legislative articles, headings, and judicial paragraphs—as splitting points (Merola & Singh, 2025; Jimeno-Yepes et al., 2024). By aligning chunks with the formal hierarchy of the law, the system is expected to maintain higher coherence and relevance, as legal units are kept intact (Nguyen et al., 2025; Tanyildiz et al., 2024). This is particularly effective for high-density documents like financial or legal reports where structure conveys meaning (Guo et al., 2025).

In [15]:
#Define splitter
structure_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len,
    # The order of separators is crucial: the first element has the highest priority.
    # We prioritize legal headers to preserve the logical integrity of the text.
    separators=[
        "\nArtikel ",      # Splits legislation cleanly by article
        "\nECLI:",         # Splits jurisprudence at the start of a new case
        "\n\n",            # Paragraphs
        "\n",              # Line breaks
        ". ",              # Sentences
        " "                # Words
    ]
)

# Apply the structure-aware splitting to the document collection
chunks_structure = structure_splitter.split_documents(documents)

print(f"Strategy 2: Structure-Aware Results")
print(f"Total chunks created: {len(chunks_structure)}")

# Inspect the first jurisprudence chunk to verify structural integrity
# We iterate to find the first sample where source_type is 'jurisprudence'
try:
    jur_sample = next(c for c in chunks_structure if c.metadata['source_type'] == 'jurisprudence')
    print(f"\nSample Structure-Aware Jurisprudence")
    print(f"Metadata: {jur_sample.metadata}")
    print(f"Content snippet: {jur_sample.page_content[:300]}...")
except StopIteration:
    print("\nNo jurisprudence chunks found. Check your input data.")

Strategy 2: Structure-Aware Results
Total chunks created: 52875

Sample Structure-Aware Jurisprudence
Metadata: {'id': 'ECLI:NL:RBGEL:2024:26', 'title': 'ECLI:NL:RBGEL:2024:26, Rechtbank Gelderland, 05-01-2024, AWB-22_5249 en 22_5252', 'source_type': 'jurisprudence'}
Content snippet: Weigering handhavingsverzoeken m.b.t. geitenhouderij. Intern salderen. Beroep gegrond vanwege een motiveringsgebrek. De rechtsgevolgen worden door de rechtbank in stand gelaten omdat in het verweerschrift afdoende is onderbouwd dat er geen sprake is van een overtreding van de Wnb.


    

Zittingspl...


### Semantic chunking
The most advanced strategy tested is semantic chunking, which groups sentences based on similarity and meaning rather than length or structure (Lu et al., 2025; Stäbler et al., 2025). Unlike fixed-size methods, semantic strategies adjust the length of the chunk to the content, creating "meaning-dense" units (Jimeno-Yepes et al., 2024; Kiss et al., 2025). Empirical evidence suggests that semantic chunking yields higher accuracy and fewer hallucinations by ensuring that each retrieved context is self-contained and semantically unified (Gomez-Cabello et al., 2025; Jadon et al., 2025).

Disadvantages:
* Expensive
* Takes a really long time
* Too big for the baseline

## Vector Store and Retrieval

In [16]:
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=embedding_deployment, 
    azure_endpoint=endpoint,              
    openai_api_version=api_version,       
    azure_ad_token_provider=token_provider
)

In [17]:
def create_vdb_in_batches(chunks, path, name, embeddings, batch_size=50):
    print(f"Creating Vector Store: {name}")
    
    # Initialize the database with the first batch
    first_batch = chunks[:batch_size]
    vector_db = Chroma.from_documents(
        documents=first_batch,
        embedding=embeddings,
        persist_directory=path
    )
    
    # Add the remaining chunks in batches
    for i in range(batch_size, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        vector_db.add_documents(documents=batch)
        print(f"Progress for {name}: {i + len(batch)}/{len(chunks)} chunks added...")
        
        # Avoid rate limits
        time.sleep(1) 
        
    print(f"{name} Vector Store successfully created at {path}\n")
    return vector_db

In [ ]:
vdb_base_path = "../data/vector_stores"

# Fixed-size baseline
vdb_fixed = create_vdb_in_batches(
    chunks=chunks_fixed, 
    path=os.path.join(vdb_base_path, "baseline_fixed"), 
    name="Fixed-Size",
    embeddings=embeddings
)

# Strcture-aware baseline
vdb_structure = create_vdb_in_batches(
    chunks=chunks_structure, 
    path=os.path.join(vdb_base_path, "baseline_structure"), 
    name="Structure-Aware",
    embeddings=embeddings
)

Creating Vector Store: Fixed-Size
Progress for Fixed-Size: 100/79725 chunks added...
Progress for Fixed-Size: 150/79725 chunks added...
Progress for Fixed-Size: 200/79725 chunks added...
Progress for Fixed-Size: 250/79725 chunks added...
Progress for Fixed-Size: 300/79725 chunks added...
Progress for Fixed-Size: 350/79725 chunks added...
Progress for Fixed-Size: 400/79725 chunks added...
Progress for Fixed-Size: 450/79725 chunks added...
Progress for Fixed-Size: 500/79725 chunks added...
Progress for Fixed-Size: 550/79725 chunks added...
Progress for Fixed-Size: 600/79725 chunks added...
Progress for Fixed-Size: 650/79725 chunks added...
Progress for Fixed-Size: 700/79725 chunks added...
Progress for Fixed-Size: 750/79725 chunks added...
Progress for Fixed-Size: 800/79725 chunks added...
Progress for Fixed-Size: 850/79725 chunks added...
Progress for Fixed-Size: 900/79725 chunks added...
Progress for Fixed-Size: 950/79725 chunks added...
Progress for Fixed-Size: 1000/79725 chunks added

## Baseline RAG pipeline

In [ ]:
vdb_path = "../data/vector_stores"

# Load fixed baseline vectorbase
vdb_fixed = Chroma(
    persist_directory=os.path.join(vdb_path, "baseline_fixed"),
    embedding_function=embeddings
)

# Load structure baseline vectorbase
vdb_structure = Chroma(
    persist_directory=os.path.join(vdb_path, "baseline_structure"),
    embedding_function=embeddings
)

print(f"Fixed-size chunks: {vdb_fixed._collection.count()}")
print(f"Structure-aware chunks: {vdb_structure._collection.count()}")

Fixed-size chunks: 81175
Structure-aware chunks: 52875


## RAGAs evaluation

### QA pairs 

In [ ]:
# Load JSON file
file_path = "../data/QA_pairs_evaluation.json" 

with open(file_path, 'r', encoding='utf-8') as f:
    qa_list = json.load(f)

# Convert to DataFrame
df_qa = pd.DataFrame(qa_list)
print(f"Dataset geladen: {len(df_qa)} vragen gevonden.")

Dataset geladen: 10 vragen gevonden.


### RAGAs dataset generations

In [ ]:
client = AsyncAzureOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
)

In [ ]:
# Strategy fixed size
fixed_retriever = vdb_fixed.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3} 
)

# Strategy structure aware
structure_retriever = vdb_structure.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
async def run_evaluation_loop(retriever, dataset_list, strategy_name): 
    questions = []
    all_contexts = []
    ground_truths = []
    all_answers = []


    system_prompt = (
    "Je bent een juridisch assistent voor de Gemeente Amsterdam.\n"
    "Beantwoord de vraag uitsluitend op basis van de verstrekte context.\n\n"
    "EISEN:\n"
    "1. Noem het specifieke wetsartikel uit de Omgevingswet.\n"
    "2. Noem het ECLI-nummer van de relevante uitspraak.\n"
    "3. Als informatie ontbreekt, geef dit dan expliciet aan."
)



    print(f"Start retrieval and generation for: {strategy_name}...")

    for item in dataset_list:
        q = item['question']
        gt = item['ground_truth']
        
        # Retrieval
        docs = retriever.invoke(q) 
        ctx_list = [doc.page_content for doc in docs]
        context_text = "\n\n".join(ctx_list)
        
        # Generation
        resp = await client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context: {context_text}\n\nVraag: {q}"},
            ],
            temperature=0 
        )
        answer = resp.choices[0].message.content
        
        # Save to lists
        questions.append(q)
        all_contexts.append(ctx_list) 
        ground_truths.append(gt)
        all_answers.append(answer)

    # Dataset object for RAGAs
    ds = Dataset.from_dict({
        "question": questions,
        "answer": all_answers,
        "contexts": all_contexts,
        "ground_truth": ground_truths
    })
    
    os.makedirs("../data/results", exist_ok=True)
    ds.to_pandas().to_csv(f"../data/results/results_{strategy_name}_6.csv", index=False, encoding='utf-16')
    
    return ds

dataset_fixed = await run_evaluation_loop(fixed_retriever, qa_list, "Fixed-Size")
dataset_struct = await run_evaluation_loop(structure_retriever, qa_list, "Structure-Aware")

Start retrieval and generation for: Fixed-Size...


CancelledError: 

## Results and analysis

In [ ]:
evaluator_llm = llm_factory(
    model=deployment,
    client=client,
    max_tokens=4096
    )

config = RunConfig(
    timeout=240,     
    max_retries=20, 
    max_wait=60,
    max_workers=1,
    seed=42,      
)

nest_asyncio.apply()

# Initialize metrics
metrics = [
    Faithfulness(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm), 
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]

# evaluate
result_fixed = await aevaluate(
    dataset=dataset_fixed,
    metrics=metrics,
    run_config=config
)

result_structure = await aevaluate(
    dataset=dataset_struct,
    metrics=metrics,
    run_config=config
)

print(f"results fixed: {result_fixed}")
print(f"results structure: {result_structure}")

C:\Users\verkad004\AppData\Local\Temp\ipykernel_16268\3627229808.py:26: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result_fixed = await aevaluate(
Evaluating: 100%|██████████| 40/40 [05:22<00:00,  8.05s/it]
C:\Users\verkad004\AppData\Local\Temp\ipykernel_16268\3627229808.py:32: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result_structure = await aevaluate(
Evaluating: 100%|██████████| 40/40 [05:18<00:00,  7.96s/it]

results fixed: {'faithfulness': 0.5526, 'factual_correctness(mode=f1)': 0.2710, 'context_precision': 0.7000, 'context_recall': 0.2800}
results structure: {'faithfulness': 0.5790, 'factual_correctness(mode=f1)': 0.2360, 'context_precision': 0.8083, 'context_recall': 0.2667}


In [ ]:
# baseline data
df_baseline = pd.read_csv('../data/results/results_Structure-Aware_5.csv',encoding='utf-16')


# print baseline data
for i in range(len(df_baseline)):
    print(f"Case {i+1}\n")
    print(f"Question:\n{df_baseline.loc[i, 'question']}\n")
    
    print(f"Answer:\n")
    print(f"{df_baseline.loc[i, 'answer']}")
    print("-" * 50)
    

Case 1

Question:
Kan een omgevingsvergunning voor een dakterras op een gemeentelijk monument worden verleend als het hekwerk de maximale bouwhoogte overschrijdt?

Answer:

Op basis van de verstrekte context is het niet mogelijk om een omgevingsvergunning te verlenen voor een dakterras op een gemeentelijk monument als het hekwerk de maximale bouwhoogte overschrijdt, tenzij er sprake is van een expliciete afwijking van het bestemmingsplan die voldoet aan de wettelijke vereisten en beleidsregels. 

In dit geval heeft het college de gevraagde omgevingsvergunning geweigerd omdat het bouwplan in strijd is met artikel 19.2.2 van de planregels van het bestemmingsplan "Museumkwartier Valeriusbuurt". De overschrijding van de maximale bouwhoogte met 0,6 meter door het hekwerk en met 2 meter door de dakopbouw is niet in overeenstemming met de toegestane bouwhoogte van 15 meter. Het college heeft bovendien geoordeeld dat er geen aanleiding is om met toepassing van artikel 2.12, eerste lid, aanhef 